<a href="https://colab.research.google.com/github/soheldatta17/mid_eval_meta_model/blob/main/Meta_Model_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# At the top of your notebook, run this once to mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os, sys, subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import tensorflow as tf
import pickle

# --- gdown ---
try:
    import gdown
except ImportError:
    subprocess.run(["pip", "install", "gdown", "-q"], check=True)
    import gdown

# ================================================================
# CONFIG
# ================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DRIVE_ASSETS = {
    "model1": {
        "id":    "1orbyJ0UU44HT3G8inoGstlJ0DhJlQXjj",
        "local": "/content/best_knee_ensemble_cbam.pt",
        "desc":  "CBAM ensemble (TorchScript)",
    },
    "model2": {
        "id":    "1Hr4gHki9nl6nmXPO0xsAU7FnlfqldHZ8",
        "local": "/content/final_knee_cnn_model.keras",
        "desc":  "Keras CNN",
    },
    "model3": {
        "id":    "16ozIZmH36J0K90bY9Jfe4YDTvS2SDDPh",
        "local": "/content/final_mmorphattention.pt",
        "desc":  "MorphAttention (TorchScript)",
    },
    "rf": {
        "id":    "1Ih1lQjV7yxYytd08a71OzEeniH4iQHxw",
        "local": "/content/meta_randomforest.pkl",
        "desc":  "RF meta-learner + scaler",
    },
    "bundle": {
        "id":    "18xJYLCe1Z-wAL5fvq8uo0yqQu91ZOOsY",
        "local": "/content/full_pipeline_bundle.pkl",
        "desc":  "Full pipeline bundle (MLP + XGBoost + scaler)",
    },
}

OPT_W1 = None
OPT_W2 = None
OPT_W3 = None

SEVERITY_NAMES = ["Normal", "Doubtful", "Mild", "Moderate", "Severe"]
NUM_SEVERITY   = 5
META_INPUT_DIM = 14

# ================================================================
# DOWNLOAD ALL ASSETS FROM DRIVE (skips if already cached)
# ================================================================
def _gdrive_download(asset_key):
    asset = DRIVE_ASSETS[asset_key]
    local = asset["local"]
    if os.path.exists(local):
        print(f"[INFO ] {asset['desc']} already cached at {local}")
        return local
    print(f"[INFO ] Downloading {asset['desc']} from Google Drive ...")
    url = f"https://drive.google.com/uc?id={asset['id']}"
    gdown.download(url, local, quiet=False)
    if not os.path.exists(local):
        raise RuntimeError(f"[ERR  ] Download failed for {asset['desc']} — check file permissions (must be 'Anyone with link')")
    size_mb = os.path.getsize(local) / 1e6
    print(f"[OK   ] Saved to {local}  ({size_mb:.1f} MB)")
    return local

print(f"\n[START] Device: {DEVICE}")
print(f"[INFO ] Checking / downloading all model assets ...\n")

for key in ["model1", "model2", "model3", "rf", "bundle"]:
    _gdrive_download(key)

# ================================================================
# LOAD SUB-MODELS
# ================================================================
print(f"\n[INFO ] Loading Model 1 (CBAM ensemble) ...")
model1 = torch.jit.load(DRIVE_ASSETS["model1"]["local"], map_location=DEVICE)
model1.eval()
print(f"[OK   ] Model 1 ready")

print(f"[INFO ] Loading Model 2 (Keras CNN) ...")
model2_tf = tf.keras.models.load_model(DRIVE_ASSETS["model2"]["local"])
model2_tf.trainable = False
try:
    _m2_last_act = model2_tf.layers[-1].activation.__name__
except Exception:
    _m2_last_act = None
MODEL2_HAS_SOFTMAX = (_m2_last_act == "softmax")
print(f"[OK   ] Model 2 ready  (final act: {_m2_last_act})")

print(f"[INFO ] Loading Model 3 (MorphAttention) ...")
model3 = torch.jit.load(DRIVE_ASSETS["model3"]["local"], map_location=DEVICE)
model3.eval()
print(f"[OK   ] Model 3 ready")

# ================================================================
# LOAD RF (old)
# ================================================================
print(f"[INFO ] Loading RF meta-learner ...")
with open(DRIVE_ASSETS["rf"]["local"], "rb") as f:
    rf_data = pickle.load(f)
    rf      = rf_data["clf"]
    scaler_rf = rf_data["scaler"]
print(f"[OK   ] RF meta-learner + scaler ready")

# ================================================================
# LOAD BUNDLE (new) — MLP + XGBoost + scaler
# ================================================================
print(f"[INFO ] Loading full_pipeline_bundle ...")
with open(DRIVE_ASSETS["bundle"]["local"], "rb") as f:
    bundle = pickle.load(f)

class IntermediateFusionMLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_dim),
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden // 2, out_dim),
        )
    def forward(self, x): return self.net(x)

cfg = bundle["mlp_config"]
mlp = IntermediateFusionMLP(cfg["in_dim"], cfg["hidden"], cfg["out_dim"])
mlp.load_state_dict(bundle["mlp_state_dict"])
mlp.eval()

xgb        = bundle["xgb"]
scaler_xgb = bundle["scaler"]
CLASS_IDS  = bundle["class_ids"]
FUSION_DIM = bundle["fusion_dim"]   # 9

print(f"[OK   ] MLP  {cfg['in_dim']}→{cfg['hidden']}→{cfg['hidden']//2}→{cfg['out_dim']}")
print(f"[OK   ] XGBoost + scaler ready")
print(f"[OK   ] Bundle version: {bundle.get('version', 'N/A')}\n")

# ================================================================
# TRANSFORMS
# ================================================================
transform_pt = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def _preprocess_pt(image_path):
    img = Image.open(image_path).convert("RGB")
    return transform_pt(img).unsqueeze(0).to(DEVICE)

def _preprocess_tf(image_path):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=1, expand_animations=False)
    img = tf.image.resize(img, [256, 256])
    img = tf.cast(img, tf.float32)
    return tf.expand_dims(img, axis=0)

# ================================================================
# CORE PREDICTION FUNCTION
# ================================================================
def predict_single(image_path, opt_w1=OPT_W1, opt_w2=OPT_W2, opt_w3=OPT_W3):
    """
    Runs both pipelines on one knee X-ray:
      Pipeline A (old) : M1+M2+M3 → 14D → RF
      Pipeline B (new) : M1+M2+M3 → Vfusion(9D) → MLP → Pmeta(5D) → XGBoost
    """
    if not os.path.exists(image_path):
        print(f"[ERR  ] File not found: {image_path}")
        return None

    print(f"\n{'='*60}")
    print(f"  IMAGE  : {os.path.basename(image_path)}")
    print(f"{'='*60}")

    # ── Sub-model inference ───────────────────────────────────────────────────
    print(f"[INFO ] Running sub-models ...")
    img_pt = _preprocess_pt(image_path)

    with torch.no_grad():
        probs1 = F.softmax(model1(img_pt), dim=1).cpu().numpy()[0]     # (5,)

    img_tf = _preprocess_tf(image_path)
    out2   = model2_tf.predict(img_tf, verbose=0)
    probs2 = out2[0] if MODEL2_HAS_SOFTMAX else tf.nn.softmax(out2).numpy()[0]
    probs2 = np.array(probs2, dtype=np.float32)                        # (5,)

    with torch.no_grad():
        raw3 = torch.sigmoid(model3(img_pt)).cpu().numpy()[0]          # (4,)
    probs3 = np.zeros(NUM_SEVERITY, dtype=np.float32)
    probs3[:len(raw3)] = raw3                                          # (5,) padded

    pred1 = int(np.argmax(probs1))
    pred2 = int(np.argmax(probs2))
    pred3 = int(np.argmax(probs3))

    print(f"\n  Per-model predictions:")
    print(f"  {'Model':<24} {'Grade':>6}  {'Label':<12} {'Confidence':>10}")
    print(f"  {'-'*56}")
    print(f"  {'Model1 CBAM ensemble':<24} {'Gr'+str(pred1):>6}  {SEVERITY_NAMES[pred1]:<12} {probs1[pred1]*100:>9.2f}%")
    print(f"  {'Model2 Keras CNN':<24} {'Gr'+str(pred2):>6}  {SEVERITY_NAMES[pred2]:<12} {probs2[pred2]*100:>9.2f}%")
    print(f"  {'Model3 MorphAttention':<24} {'Gr'+str(pred3):>6}  {SEVERITY_NAMES[pred3]:<12} {probs3[pred3]*100:>9.2f}%")

    # ── Weighted ensemble (display only) ─────────────────────────────────────
    if opt_w1 is not None and opt_w2 is not None and opt_w3 is not None:
        w1, w2, w3 = opt_w1, opt_w2, opt_w3
        weight_src = "optimized"
    else:
        w1 = w2 = w3 = 1/3
        weight_src = "equal"

    ensemble_probs = w1*probs1 + w2*probs2 + w3*probs3
    ensemble_pred  = int(np.argmax(ensemble_probs))

    print(f"\n  Weighted ensemble ({weight_src}: M1={w1:.2f} M2={w2:.2f} M3={w3:.2f}):")
    print(f"  {'Grade':<6} {'Label':<12} {'Prob':>8}  Bar")
    print(f"  {'-'*46}")
    for i, (name, prob) in enumerate(zip(SEVERITY_NAMES, ensemble_probs)):
        bar    = "█" * int(prob * 35)
        marker = "  ← ensemble pred" if i == ensemble_pred else ""
        print(f"  Gr{i:<4} {name:<12} {prob*100:>7.2f}%  {bar}{marker}")

    # ════════════════════════════════════════════════════════════════
    # PIPELINE A — RF meta-learner  (old, 14D)
    # ════════════════════════════════════════════════════════════════
    print(f"\n{'─'*60}")
    print(f"  PIPELINE A — RF Meta-Learner  (14D: 5+5+4)")
    print(f"{'─'*60}")

    feat_vec_rf = np.concatenate([probs1, probs2, raw3]).astype(np.float32).reshape(1, -1)
    feat_sc_rf  = scaler_rf.transform(feat_vec_rf)
    rf_pred     = int(rf.predict(feat_sc_rf)[0])
    rf_proba    = rf.predict_proba(feat_sc_rf)[0]

    print(f"  {'Grade':<6} {'Label':<12} {'Prob':>8}  Bar")
    print(f"  {'-'*46}")
    for i, (name, prob) in enumerate(zip(SEVERITY_NAMES, rf_proba)):
        bar    = "█" * int(prob * 35)
        marker = "  ← RF pred" if i == rf_pred else ""
        print(f"  Gr{i:<4} {name:<12} {prob*100:>7.2f}%  {bar}{marker}")

    print(f"\n  RF Final  → Grade {rf_pred}  |  {SEVERITY_NAMES[rf_pred]}"
          f"  |  conf {rf_proba[rf_pred]*100:.2f}%")

    # ════════════════════════════════════════════════════════════════
    # PIPELINE B — MLP + XGBoost  (new, 9D → 14D)
    # ════════════════════════════════════════════════════════════════
    print(f"\n{'─'*60}")
    print(f"  PIPELINE B — MLP + XGBoost Arbiter  (Vfusion 9D → Xboost 14D)")
    print(f"{'─'*60}")

    # Build Vfusion (9D): sev(5) + jsn(2) + morph(2)
    jsn = np.array(
        [float(probs2[2]+probs2[3]+probs2[4]),   # P(narrow) = grade >= 2
         float(probs2[0]+probs2[1])],             # P(normal) = grade < 2
        dtype=np.float32
    )
    morph2  = raw3[:2].astype(np.float32)         # Osteophytes, Sclerosis
    vfusion = np.concatenate([probs1, jsn, morph2])   # (9,)

    # MLP → Pmeta (5D)
    x_tensor = torch.tensor(vfusion, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        pmeta = F.softmax(mlp(x_tensor), dim=1).numpy()[0]             # (5,)
    mlp_pred = int(np.argmax(pmeta))

    print(f"  Vfusion (9D): sev={np.round(probs1,3)}  "
          f"jsn={np.round(jsn,3)}  morph={np.round(morph2,3)}")
    print(f"  Pmeta  (5D): {np.round(pmeta,3)}  → MLP pred G{mlp_pred}")

    # XGBoost on Xboost (14D)
    xboost_vec = scaler_xgb.transform(
        np.hstack([vfusion, pmeta])[None]
    )
    xgb_pred  = int(xgb.predict(xboost_vec)[0])
    xgb_proba = xgb.predict_proba(xboost_vec)[0]
    xgb_grade = CLASS_IDS[xgb_pred]

    print(f"\n  {'Grade':<6} {'Label':<12} {'Prob':>8}  Bar")
    print(f"  {'-'*46}")
    for i, (name, prob) in enumerate(zip(SEVERITY_NAMES, xgb_proba)):
        bar    = "█" * int(prob * 35)
        marker = "  ← XGB pred" if i == xgb_pred else ""
        print(f"  Gr{i:<4} {name:<12} {prob*100:>7.2f}%  {bar}{marker}")

    print(f"\n  XGB Final → Grade {xgb_grade}  |  {SEVERITY_NAMES[xgb_pred]}"
          f"  |  conf {xgb_proba[xgb_pred]*100:.2f}%")

    # Clinical signals
    print(f"\n  Clinical signals (from Vfusion):")
    print(f"    JSN        : {'⚠ NARROWING' if jsn[0]>=0.5 else 'normal'}"
          f"  (narrow={jsn[0]*100:.1f}%  normal={jsn[1]*100:.1f}%)")
    print(f"    Osteophyte : {'⚠ POSITIVE' if morph2[0]>0.5 else 'negative'}"
          f"  ({morph2[0]*100:.1f}%)")
    print(f"    Sclerosis  : {'⚠ POSITIVE' if morph2[1]>0.5 else 'negative'}"
          f"  ({morph2[1]*100:.1f}%)")

    # ── Agreement across ALL predictors ───────────────────────────────────────
    all_preds   = [pred1, pred2, pred3, ensemble_pred, rf_pred, mlp_pred, xgb_pred]
    agree_count = all_preds.count(xgb_pred)
    agreement   = "HIGH" if agree_count >= 5 else "MODERATE" if agree_count >= 3 else "LOW"

    print(f"\n{'='*60}")
    print(f"  ★  FINAL PREDICTION  (XGBoost Arbiter — Pipeline B)")
    print(f"{'='*60}")
    print(f"  KL Grade          : {xgb_grade}")
    print(f"  Severity          : {SEVERITY_NAMES[xgb_pred]}")
    print(f"  XGB confidence    : {xgb_proba[xgb_pred]*100:.2f}%")
    print(f"  RF  confidence    : {rf_proba[rf_pred]*100:.2f}%  (G{rf_pred})")
    print(f"  Agreement         : {agreement}  ({agree_count}/7 → G{xgb_grade})")
    print(f"  {'─'*40}")
    print(f"  M1 pred    : G{pred1}  {SEVERITY_NAMES[pred1]}")
    print(f"  M2 pred    : G{pred2}  {SEVERITY_NAMES[pred2]}")
    print(f"  M3 pred    : G{pred3}  {SEVERITY_NAMES[pred3]}")
    print(f"  Ensemble   : G{ensemble_pred}  {SEVERITY_NAMES[ensemble_pred]}")
    print(f"  RF         : G{rf_pred}  {SEVERITY_NAMES[rf_pred]}")
    print(f"  MLP meta   : G{mlp_pred}  {SEVERITY_NAMES[mlp_pred]}")
    print(f"  XGB final  : G{xgb_grade}  {SEVERITY_NAMES[xgb_pred]}  ✓")
    print(f"{'='*60}\n")

    return {
        # ── final answer ──
        "grade"           : xgb_grade,
        "label"           : SEVERITY_NAMES[xgb_pred],
        "confidence"      : float(xgb_proba[xgb_pred]),
        # ── pipeline B ──
        "xgb_proba"       : xgb_proba.tolist(),
        "mlp_proba"       : pmeta.tolist(),
        "jsn_narrow"      : round(float(jsn[0]), 3),
        "jsn_normal"      : round(float(jsn[1]), 3),
        "osteophyte"      : round(float(morph2[0]), 3),
        "sclerosis"       : round(float(morph2[1]), 3),
        # ── pipeline A ──
        "rf_grade"        : rf_pred,
        "rf_confidence"   : float(rf_proba[rf_pred]),
        "rf_proba"        : rf_proba.tolist(),
        # ── sub-models ──
        "ensemble_probs"  : ensemble_probs.tolist(),
        "ensemble_pred"   : ensemble_pred,
        "model1_pred"     : pred1,
        "model2_pred"     : pred2,
        "model3_pred"     : pred3,
        "agreement"       : agreement,
    }


[START] Device: cuda
[INFO ] Checking / downloading all model assets ...

[INFO ] CBAM ensemble (TorchScript) already cached at /content/best_knee_ensemble_cbam.pt
[INFO ] Keras CNN already cached at /content/final_knee_cnn_model.keras
[INFO ] MorphAttention (TorchScript) already cached at /content/final_mmorphattention.pt
[INFO ] RF meta-learner + scaler already cached at /content/meta_randomforest.pkl

[INFO ] Loading Model 1 (CBAM ensemble) ...
[OK  ] Model 1 ready
[INFO ] Loading Model 2 (Keras CNN) ...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


[OK  ] Model 2 ready  (final act: softmax)
[INFO ] Loading Model 3 (MorphAttention) ...
[OK  ] Model 3 ready
[INFO ] Loading RF meta-learner ...
[OK  ] RF meta-learner + scaler ready


In [7]:


# ================================================================
# RUN
# ================================================================
IMAGE_PATH = "/content/2.png"

result = predict_single(IMAGE_PATH)


  IMAGE  : 2.png
[INFO ] Running Model 1 ...
[INFO ] Running Model 2 ...
[INFO ] Running Model 3 ...

  Per-model predictions:
  Model                     Grade  Label        Confidence
  ------------------------------------------------------
  Model1 CBAM ensemble        Gr1  Doubtful         99.93%
  Model2 Keras CNN            Gr4  Severe          100.00%
  Model3 MorphAttention       Gr1  Doubtful         52.21%

  Ensemble weights (equal (no optimized weights set)):
  M1=0.333  M2=0.333  M3=0.333

  Weighted ensemble probability distribution:
  Grade  Label            Prob  Bar
  --------------------------------------------
  Gr0    Normal         16.72%  ######
  Gr1    Doubtful       50.71%  ####################  <-- predicted
  Gr2    Mild           16.61%  ######
  Gr3    Moderate       16.58%  ######
  Gr4    Severe         33.33%  #############

[INFO ] Running RF meta-learner ...

  RF meta-learner prediction:
  Grade  Label            Prob  Bar
  -------------------------